# Bart Project

In [35]:
import pandas as pd
import math

## EDA - Exploraty Data Analysis

In [2]:
df1=pd.read_csv("date-hour-soo-dest-2016.csv")

In [3]:
df2=pd.read_csv("date-hour-soo-dest-2017.csv")

In [4]:
df3=pd.read_csv("station_info.csv")

In [5]:
df1

,Origin,Destination,Throughput,DateTime
0,12TH,12TH,1,2016-01-01 00:00:00
1,12TH,16TH,1,2016-01-01 00:00:00
2,12TH,24TH,4,2016-01-01 00:00:00
3,12TH,ASHB,4,2016-01-01 00:00:00
4,12TH,BALB,2,2016-01-01 00:00:00
...,...,...,...,...
9971577,WOAK,PLZA,1,2016-12-31 23:00:00
9971578,WOAK,POWL,30,2016-12-31 23:00:00
9971579,WOAK,RICH,2,2016-12-31 23:00:00
9971580,WOAK,ROCK,2,2016-12-31 23:00:00


In [6]:
df1.isnull().sum()

Origin         0
Destination    0
Throughput     0
DateTime       0
dtype: int64

In [7]:
df2.isnull().sum()

Origin         0
Destination    0
Throughput     0
DateTime       0
dtype: int64

In [8]:
df3.isnull().sum()

Abbreviation    0
Description     0
Location        0
Name            0
dtype: int64

In [9]:
df1.Throughput.value_counts()

Throughput
1       2388225
2       1460216
3        987580
4        715681
5        542845
         ...   
851           1
891           1
1075          1
1039          1
884           1
Name: count, Length: 915, dtype: int64

## Data Preparation and Feature Engineering

In [10]:
df=pd.concat([df1,df2],ignore_index=True)

In [11]:
df.head()

,Origin,Destination,Throughput,DateTime
0,12TH,12TH,1,2016-01-01 00:00:00
1,12TH,16TH,1,2016-01-01 00:00:00
2,12TH,24TH,4,2016-01-01 00:00:00
3,12TH,ASHB,4,2016-01-01 00:00:00
4,12TH,BALB,2,2016-01-01 00:00:00


In [12]:
df = df.merge(df3, left_on='Origin', right_on='Abbreviation', how='left')
df.rename(columns={'Name': 'Origin_Name', 'Description': 'Origin_Description', 'Location': 'Origin_Location'}, inplace=True)
df.drop(columns=["Abbreviation"],inplace=True)

In [13]:
df = df.merge(df3, left_on='Destination', right_on='Abbreviation', how='left')
df.rename(columns={
    'Name': 'Dest_Name', 
    'Description': 'Dest_Description', 
    'Location': 'Dest_Location'
}, inplace=True)
df.drop(columns=['Abbreviation'], inplace=True)

In [14]:
df.columns

Index(['Origin', 'Destination', 'Throughput', 'DateTime', 'Origin_Description',
       'Origin_Location', 'Origin_Name', 'Dest_Description', 'Dest_Location',
       'Dest_Name'],
      dtype='str')

In [15]:
df3.columns

Index(['Abbreviation', 'Description', 'Location', 'Name'], dtype='str')

In [16]:
df['DateTime'] = pd.to_datetime(df['DateTime'])

In [19]:
split_origin = df['Origin_Location'].str.split(',', expand=True)
df['Origin_Lat'] = split_origin[0].astype(float)
df['Origin_Lon'] = split_origin[1].astype(float)

split_dest = df['Dest_Location'].str.split(',', expand=True)
df['Dest_Lat'] = split_dest[0].astype(float)
df['Dest_Lon'] = split_dest[1].astype(float)

In [20]:
df.head()

,Origin,Destination,Throughput,DateTime,Origin_Description,Origin_Location,Origin_Name,Dest_Description,Dest_Location,Dest_Name,Origin_Lat,Origin_Lon,Dest_Lat,Dest_Lon
0,12TH,12TH,1,2016-01-01,"1245 Broadway, Oakland CA 94612<br />12th St. ...","-122.271450,37.803768,0",12th St. Oakland City Center (12TH),"1245 Broadway, Oakland CA 94612<br />12th St. ...","-122.271450,37.803768,0",12th St. Oakland City Center (12TH),-122.27145,37.803768,-122.271450,37.803768
1,12TH,16TH,1,2016-01-01,"1245 Broadway, Oakland CA 94612<br />12th St. ...","-122.271450,37.803768,0",12th St. Oakland City Center (12TH),"2000 Mission Street, San Francisco CA 94110<br...","-122.419694,37.765062,0",16th St. Mission (16TH),-122.27145,37.803768,-122.419694,37.765062
2,12TH,24TH,4,2016-01-01,"1245 Broadway, Oakland CA 94612<br />12th St. ...","-122.271450,37.803768,0",12th St. Oakland City Center (12TH),"2800 Mission Street, San Francisco CA 94110<br...","-122.418143,37.752470,0",24th St. Mission (24TH),-122.27145,37.803768,-122.418143,37.752470
3,12TH,ASHB,4,2016-01-01,"1245 Broadway, Oakland CA 94612<br />12th St. ...","-122.271450,37.803768,0",12th St. Oakland City Center (12TH),"3100 Adeline Street, Berkeley CA 94703<br />As...","-122.270062,37.852803,0",Ashby (ASHB),-122.27145,37.803768,-122.270062,37.852803
4,12TH,BALB,2,2016-01-01,"1245 Broadway, Oakland CA 94612<br />12th St. ...","-122.271450,37.803768,0",12th St. Oakland City Center (12TH),"401 Geneva Avenue, San Francisco CA 94112<br /...","-122.447506,37.721585,0",Balboa Park (BALB),-122.27145,37.803768,-122.447506,37.721585


## Data Analytics

In [21]:
origin_totals = df.groupby('Origin_Name')['Throughput'].sum()
dest_totals = df.groupby('Dest_Name')['Throughput'].sum()

In [22]:
busiest_stations = (origin_totals + dest_totals).sort_values(ascending=False)

In [23]:
busiest_stations.head(10)

Origin_Name
Embarcadero (EMBR)                     34076644
Montgomery St. (MONT)                  33063895
Powell St. (POWL)                      26593039
Civic Center/UN Plaza (CIVC)           19536703
Downtown Berkeley (DBRK)               10806360
24th St. Mission (24TH)                10762588
16th St. Mission (16TH)                10749505
12th St. Oakland City Center (12TH)    10714765
19th St. Oakland (19TH)                10392910
Balboa Park (BALB)                      9070671
Name: Throughput, dtype: int64

In [24]:
route_totals = df.groupby(['Origin_Name', 'Dest_Name'])['Throughput'].sum().sort_values(ascending=True)

In [25]:
route_totals

Origin_Name                    Dest_Name                    
West Dublin/Pleasanton (WDUB)  North Concord/Martinez (NCON)        358
North Concord/Martinez (NCON)  West Dublin/Pleasanton (WDUB)        358
Lafayette (LAFY)               Castro Valley (CAST)                 444
West Dublin/Pleasanton (WDUB)  Orinda (ORIN)                        645
                               Lafayette (LAFY)                     660
                                                                 ...   
Dublin/Pleasanton (DUBL)       Embarcadero (EMBR)                901632
Powell St. (POWL)              24th St. Mission (24TH)           913309
Balboa Park (BALB)             Powell St. (POWL)                 938212
                               Montgomery St. (MONT)            1013860
Powell St. (POWL)              Balboa Park (BALB)               1105884
Name: Throughput, Length: 2025, dtype: int64

In [26]:
df['Hour'] = df['DateTime'].dt.hour
berkeley_df = df[df['Origin'] == 'DBRK']
hourly_seats = berkeley_df.groupby('Hour')['Throughput'].sum().sort_values(ascending=True)

In [30]:
hourly_seats.head()

Hour
3       48
2      240
4     1758
5    13074
1    13528
Name: Throughput, dtype: int64

In [31]:
df['Day_of_Week'] = df['DateTime'].dt.day_name()
daily_ridership = df.groupby('Day_of_Week')['Throughput'].sum().sort_values(ascending=False)

In [32]:
daily_ridership

Day_of_Week
Wednesday    30677189
Tuesday      30350159
Thursday     30049974
Friday       28373709
Monday       26967803
Saturday     13696288
Sunday        9602140
Name: Throughput, dtype: int64

In [33]:
late_night_hours = [23, 0, 1, 2, 3, 4]
late_night_df = df[df['Hour'].isin(late_night_hours)]
total_late_night_riders = late_night_df['Throughput'].sum()

In [34]:
total_late_night_riders

np.int64(5108501)

In [39]:
stations = df[['Origin_Name', 'Origin_Lat', 'Origin_Lon']].drop_duplicates().dropna().reset_index(drop=True)

In [40]:
def basic_distance(lat1, lon1, lat2, lon2):
# Converting latitude and longitude differences to kilometers (~111 km/degree)   
    d_lat = (lat2 - lat1) * 111
    d_lon = (lon2 - lon1) * 111
    
    return math.sqrt(d_lat**2 + d_lon**2)

In [41]:
st1 = stations.iloc[0]
st2 = stations.iloc[1]

distance = basic_distance(st1['Origin_Lat'], st1['Origin_Lon'], st2['Origin_Lat'], st2['Origin_Lon'])

In [42]:
distance

17.006720737785677